# Phase-6: Supervised Predictive Validation & Tiering (London-final-light)

Two things, both grounded in a real temporal hold-out (no leakage):
1. **Validate** the risk index: does a score built on the past predict the future? (ROC / AUC vs a baseline)
2. **Build the tiers** from the prediction itself: instead of K-means, derive each tier boundary from
   its own ROC test, so all four tiers are evidence-based.

**Hold-out:** train = first 24 months (Apr 2023 to Mar 2025); ground truth = actual crime in the
final 12 months (Apr 2025 to Mar 2026).

**Reads:** `london_midterm/outputs/phase1`+`phase2`, `data/lsoa_nightlights_england.parquet`,
`london-final-light/outputs/phase5` (K-means k=4 for comparison). **Writes:** `.../outputs/phase6/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, classification_report
print('Imports OK')

In [ ]:
# Dataset Loading
BASE = Path('Dataset Path')
P1   = BASE / 'london_midterm' / 'outputs' / 'phase1'
P2   = BASE / 'london_midterm' / 'outputs' / 'phase2'
P5   = BASE / 'london-final-light' / 'outputs' / 'phase5'
OUT  = BASE / 'london-final-light' / 'outputs' / 'phase6'
OUT.mkdir(parents=True, exist_ok=True)
SHP_DIR = BASE / 'data' / 'LB_shp'

SPLIT   = pd.Timestamp('2025-04-01')   # first 24 months < SPLIT ; final 12 months >= SPLIT
# Future-demand levels that define the tier boundaries (top X% of future crime)
TIER_LEVELS = [0.15, 0.40, 0.70]       # -> Tier1 / Tier2 / Tier3 lines ; rest = Tier4
LABEL_PCT   = 0.15                      # headline 'high demand' = top 15% for the AUC validation

WEIGHTS = {'severity_weighted_count': 0.7670, 'ntl_mean_radiance': 0.1302,
           'employment_deprivation': 0.0775, 'resolution_rate': 0.0253}   # Phase 4 blended weights
LOG_FEATURES = ['severity_weighted_count', 'ntl_mean_radiance']

CCHI_WEIGHTS = {
    'Violence and sexual offences': 670, 'Robbery': 365, 'Burglary': 281, 'Vehicle crime': 6,
    'Theft from the person': 2, 'Shoplifting': 1, 'Other theft': 4, 'Bicycle theft': 5,
    'Criminal damage and arson': 98, 'Drugs': 156, 'Public order': 53,
    'Possession of weapons': 541, 'Other crime': 74, 'Anti-social behaviour': 1}
RESOLVED_OUTCOMES = {
    'Suspect charged', 'Offender given a caution', 'Offender given a penalty notice',
    'Offender fined', 'Offender deported', 'Offender otherwise dealt with',
    'Suspect charged as part of another case', 'Local resolution',
    'Offender given a drugs possession warning', 'Offender given conditional discharge',
    'Offender given absolute discharge', 'Offender sent to prison',
    'Offender given suspended prison sentence', 'Offender given community sentence'}
TIER_COLOURS = {1: '#D62728', 2: '#FF7F0E', 3: '#2CA02C', 4: '#1F77B4'}
report = []
print('Config loaded.')

## 1. Load data: build static features

In [ ]:
crimes   = pd.read_parquet(P1 / 'phase1_crimes_london.parquet',
                           columns=['Crime ID', 'Month', 'Crime type', 'LSOA code'])
outcomes = pd.read_parquet(P1 / 'phase1_outcomes_london.parquet', columns=['Crime ID', 'Outcome type'])
fm       = pd.read_parquet(P2 / 'phase2_feature_matrix.parquet')
ntl      = pd.read_parquet(BASE / 'data' / 'lsoa_nightlights_england.parquet')[['lsoa21cd', 'ntl_mean_radiance']]

LSOAS  = fm[['lsoa21cd']].copy()
crimes = crimes[crimes['LSOA code'].isin(set(LSOAS['lsoa21cd']))].rename(columns={'LSOA code': 'lsoa21cd'})

static = fm[['lsoa21cd', 'employment_rank']].merge(ntl, on='lsoa21cd', how='left')
mr = static['employment_rank'].max()
static['employment_deprivation'] = mr + 1 - static['employment_rank']
static['ntl_mean_radiance'] = static['ntl_mean_radiance'].fillna(static['ntl_mean_radiance'].median())
print(f'Crimes: {len(crimes):,} | LSOAs: {len(LSOAS):,} | range {crimes["Month"].min():%b %Y}..{crimes["Month"].max():%b %Y}')

## 2. Temporal split

In [ ]:
train  = crimes[crimes['Month'] <  SPLIT].copy()
future = crimes[crimes['Month'] >= SPLIT].copy()
print(f'Train : {train["Month"].min():%b %Y}..{train["Month"].max():%b %Y} ({len(train):,})')
print(f'Future: {future["Month"].min():%b %Y}..{future["Month"].max():%b %Y} ({len(future):,})')
report += [f'Train 24mo: {len(train):,} crimes', f'Future 12mo: {len(future):,} crimes']

## 3. Rebuild the risk score on the 24-month window only

In [ ]:
train = train.copy()
train['cchi'] = train['Crime type'].map(CCHI_WEIGHTS).fillna(74)
agg = train.groupby('lsoa21cd').agg(crime_count_24=('Crime ID', 'count'),
                                    severity_weighted_count=('cchi', 'sum')).reset_index()
tw = train[train['Crime ID'].notna()][['Crime ID', 'lsoa21cd']].merge(
        outcomes[outcomes['Crime ID'].notna()], on='Crime ID', how='left')
tw['resolved'] = tw['Outcome type'].apply(
        lambda x: any(r.lower() in str(x).lower() for r in RESOLVED_OUTCOMES) if pd.notna(x) else False)
res = tw.groupby('lsoa21cd').agg(n=('Crime ID', 'count'), r=('resolved', 'sum')).reset_index()
res['resolution_rate'] = (res['r'] / res['n'] * 100).round(2)

d = (LSOAS.merge(agg, on='lsoa21cd', how='left')
          .merge(res[['lsoa21cd', 'resolution_rate']], on='lsoa21cd', how='left')
          .merge(static[['lsoa21cd', 'employment_deprivation', 'ntl_mean_radiance']], on='lsoa21cd', how='left'))
for c in ['crime_count_24', 'severity_weighted_count']:
    d[c] = d[c].fillna(0)
d['resolution_rate'] = d['resolution_rate'].fillna(d['resolution_rate'].mean())

FEATURES = list(WEIGHTS.keys())
X = d[FEATURES].copy()
for c in LOG_FEATURES:
    X[c] = np.log1p(X[c])
Xn = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=FEATURES, index=d.index)
d['risk_score_24'] = sum(Xn[f] * WEIGHTS[f] for f in FEATURES)
d['risk_score_24'] = ((d['risk_score_24'] - d['risk_score_24'].min()) /
                      (d['risk_score_24'].max() - d['risk_score_24'].min()) * 100).round(2)
print('24-month risk score rebuilt for', len(d), 'LSOAs')

## 4. Ground-truth label (future window) + 5. validation vs baseline

In [ ]:
fut = future.groupby('lsoa21cd').size().reset_index(name='crime_count_future')
d = d.merge(fut, on='lsoa21cd', how='left')
d['crime_count_future'] = d['crime_count_future'].fillna(0)
d['high_demand'] = (d['crime_count_future'] >= d['crime_count_future'].quantile(1 - LABEL_PCT)).astype(int)

y = d['high_demand']
auc_index    = roc_auc_score(y, d['risk_score_24'])
auc_baseline = roc_auc_score(y, d['crime_count_24'])
print('--- ROC-AUC: predicting future top-15% crime LSOAs ---')
print(f'  Composite risk index (24mo): {auc_index:.4f}')
print(f'  Baseline: past crime count : {auc_baseline:.4f}')
print(f'  Index vs baseline: {auc_index-auc_baseline:+.4f}')
report += [f'AUC index={auc_index:.4f}', f'AUC baseline(past crime)={auc_baseline:.4f}']

## 6. Derive the four tiers from THREE ROC-validated cut-points
For each future-demand level (top 15% / 40% / 70%) we run a separate ROC test and take the
Youden's J optimal cut. Each boundary is therefore the score value that best separates that level
of real future demand. The three cuts split the LSOAs into four evidence-based tiers.

In [ ]:
cuts, aucs, stats = {}, {}, {}
for L in TIER_LEVELS:
    yL = (d['crime_count_future'] >= d['crime_count_future'].quantile(1 - L)).astype(int)
    fpr, tpr, thr = roc_curve(yL, d['risk_score_24'])
    jstat = tpr - fpr
    k = int(np.argmax(jstat))
    cuts[L] = float(thr[k]); aucs[L] = roc_auc_score(yL, d['risk_score_24'])
    stats[L] = (tpr[k], 1 - fpr[k])
    print(f'  top-{int(L*100):>2}% future crime: AUC={aucs[L]:.3f}  ->  Youden cut score>= {cuts[L]:.1f}  '
          f'(recall={tpr[k]:.2f}, spec={1-fpr[k]:.2f})')

t1, t2, t3 = cuts[0.15], cuts[0.40], cuts[0.70]
if not (t1 >= t2 >= t3):
    print('  NOTE: Youden cuts not monotonic; sorting so tiers are properly nested.')
    t1, t2, t3 = sorted([t1, t2, t3], reverse=True)
print(f'\nTier boundaries (score): Tier1 >= {t1:.1f} | Tier2 >= {t2:.1f} | Tier3 >= {t3:.1f} | else Tier4')
report += [f'ROC tier cuts: T1>={t1:.1f}, T2>={t2:.1f}, T3>={t3:.1f}',
           f'AUC by level: 15%={aucs[0.15]:.3f}, 40%={aucs[0.40]:.3f}, 70%={aucs[0.70]:.3f}']

In [ ]:
def sup_tier(s):
    if s >= t1: return 1
    if s >= t2: return 2
    if s >= t3: return 3
    return 4
d['supervised_tier'] = d['risk_score_24'].apply(sup_tier)
sizes = d['supervised_tier'].value_counts().sort_index()
print('Supervised tier sizes (1=highest demand):')
for t in range(1, 5):
    n = int(sizes.get(t, 0))
    share_hd = d.loc[d['supervised_tier'] == t, 'high_demand'].mean() if n else 0
    print(f'  Tier {t}: {n:>4} LSOAs ({n/len(d)*100:4.1f}%)  | share that are true top-15% hotspots: {share_hd*100:4.1f}%')
    report.append(f'Tier{t}: n={n} ({n/len(d)*100:.1f}%), hotspot-share={share_hd*100:.1f}%')

## 7. ROC curve with the three tier cut-points

In [ ]:
matplotlib.rcParams['figure.dpi'] = 120
y15 = (d['crime_count_future'] >= d['crime_count_future'].quantile(0.85)).astype(int)
fpr, tpr, thr = roc_curve(y15, d['risk_score_24'])
fpr_b, tpr_b, _ = roc_curve(y15, d['crime_count_24'])
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color='#D85A30', lw=2, label=f'Risk index, top-15% test (AUC={auc_index:.2f})')
ax.plot(fpr_b, tpr_b, color='#1D9E75', lw=1.8, ls='-.', label=f'Baseline past crime (AUC={auc_baseline:.2f})')
ax.plot([0,1],[0,1], color='navy', lw=1.3, ls='--', label='Random (0.50)')
for L, col in zip(TIER_LEVELS, ['#D62728', '#FF7F0E', '#2CA02C']):
    yL = (d['crime_count_future'] >= d['crime_count_future'].quantile(1 - L)).astype(int)
    f2, t2_, th2 = roc_curve(yL, d['risk_score_24'])
    k = int(np.argmax(t2_ - f2))
    ax.scatter(f2[k], t2_[k], color=col, s=80, zorder=5, label=f'Tier cut (top {int(L*100)}%) score>= {th2[k]:.0f}')
ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate')
ax.set_title('Phase 6: ROC-validated tier cut-points (temporal hold-out)')
ax.legend(loc='lower right', fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / '6_roc_tier_cuts.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: 6_roc_tier_cuts.png')

## 8. Compare the supervised tiers with the K-means (k=4) tiers

In [ ]:
km = pd.read_parquet(P5 / 'phase5_clusters_k4.parquet')[['lsoa21cd', 'tier']].rename(columns={'tier': 'kmeans_tier'})
cmp = d[['lsoa21cd', 'supervised_tier', 'risk_score_24', 'high_demand']].merge(km, on='lsoa21cd', how='inner')
ct = pd.crosstab(cmp['supervised_tier'], cmp['kmeans_tier'])
print('Cross-tab: supervised tier (rows) vs K-means k=4 tier (cols), 1=highest\n')
print(ct.to_string())
agree = (cmp['supervised_tier'] == cmp['kmeans_tier']).mean()
within1 = ((cmp['supervised_tier'] - cmp['kmeans_tier']).abs() <= 1).mean()
print(f'\nExact tier agreement:   {agree*100:.1f}%')
print(f'Within +/-1 tier:       {within1*100:.1f}%')
print('(Caveat: K-means used the full 36-month score; supervised uses the 24-month score + future label.)')
report += [f'Supervised vs K-means k4 agreement: {agree*100:.1f}% exact, {within1*100:.1f}% within 1 tier']

## 9. Map the supervised tiers

In [ ]:
gdf = pd.concat([gpd.read_file(f) for f in SHP_DIR.glob('*.shp')], ignore_index=True)
gdf = gpd.GeoDataFrame(gdf, crs=gpd.read_file(list(SHP_DIR.glob('*.shp'))[0]).crs)
g = gdf.merge(d[['lsoa21cd', 'supervised_tier']], on='lsoa21cd', how='left').to_crs(epsg=4326)
fig, ax = plt.subplots(figsize=(13, 9))
labels = {1: 'Tier 1 - High (top 15% cut)', 2: 'Tier 2 - Elevated (top 40%)',
          3: 'Tier 3 - Moderate (top 70%)', 4: 'Tier 4 - Low'}
for t in range(1, 5):
    g[g['supervised_tier'] == t].plot(ax=ax, color=TIER_COLOURS[t], linewidth=0.05, edgecolor='white', alpha=0.85)
um = g[g['supervised_tier'].isna()]
if len(um): um.plot(ax=ax, color='#cccccc', linewidth=0.05, edgecolor='white')
ax.legend(handles=[mpatches.Patch(color=TIER_COLOURS[t], label=labels[t]) for t in range(1, 5)],
          loc='lower left', fontsize=9, framealpha=0.9)
ax.set_title('Supervised (ROC-validated) priority tiers - london-final-light', fontsize=15, fontweight='bold')
ax.set_axis_off(); plt.tight_layout()
plt.savefig(OUT / '6_supervised_tier_map.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: 6_supervised_tier_map.png')

## 10. Save outputs

In [ ]:
keep = ['lsoa21cd', 'crime_count_24', 'severity_weighted_count', 'resolution_rate',
        'employment_deprivation', 'ntl_mean_radiance', 'risk_score_24',
        'crime_count_future', 'high_demand', 'supervised_tier']
d[keep].to_parquet(OUT / 'phase6_supervised_tiers.parquet', index=False)
print('Saved phase6_supervised_tiers.parquet', d[keep].shape)

lines = ['CBL-16 Phase 6 - Supervised Validation & Tiering (london-final-light)', '='*62] + report
(OUT / 'phase6_report.txt').write_text('\n'.join(lines), encoding='utf-8')
print('Saved phase6_report.txt')
print('\n' + '\n'.join(lines))